In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="flooit-analytics-project")

In [3]:
import pandas as pd

In [4]:
day_0 = pd.read_csv('day_0.csv', parse_dates = ['day_0'])

print(day_0.shape)
print(day_0.dtypes)
display(day_0.head())

(4319, 2)
user_pseudo_id            object
day_0             datetime64[ns]
dtype: object


,user_pseudo_id,day_0
0,1814A35D096333518F94B02DDFE3BFEC,2018-06-15
1,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15
2,6B41795D5E5B7339E007330941C9E201,2018-06-15
3,0FB42D5FFB79A9DE147873753BF7A664,2018-07-15
4,61C77C8D9E7F92524289DA7E9D4786BF,2018-07-15


In [6]:
query = """WITH params AS (SELECT user_pseudo_id,
                       event_date,
                       event_timestamp,
                       event_name,
                       param.key AS parameter_name,
                       COALESCE(param.value.string_value,
                                CAST(param.value.int_value    AS STRING),
                                CAST(param.value.float_value  AS STRING),
                                CAST(param.value.double_value AS STRING)) AS param_value
                FROM `firebase-public-project.analytics_153293282.events_*` CROSS JOIN UNNEST(event_params) AS param
                WHERE event_name IN ('in_app_purchase', 'ad_reward')),

     pivoted AS (SELECT user_pseudo_id, event_date, event_timestamp, event_name,
                     MAX(IF(parameter_name = 'product_id', param_value, NULL)) AS product_id,
                     MAX(IF(parameter_name = 'price', param_value, NULL)) AS price,
                     MAX(IF(parameter_name = 'currency', param_value, NULL)) AS currency,
                     MAX(IF(parameter_name = 'validated', param_value, NULL)) AS validated,
                     MAX(IF(parameter_name = 'type', param_value, NULL)) AS type,
                     MAX(IF(parameter_name = 'value', param_value, NULL)) AS value
                 FROM params
                 GROUP BY user_pseudo_id, event_date, event_timestamp, event_name)

SELECT user_pseudo_id, event_date, event_timestamp, event_name, product_id, SAFE_CAST(price AS FLOAT64)/1000000 AS price, currency, validated, type, SAFE_CAST(value AS FLOAT64) AS value
FROM pivoted
ORDER BY user_pseudo_id, event_timestamp"""
mon = client.query(query).to_dataframe()
print(mon.shape)

(1939, 10)


In [7]:
mon.to_csv("monetization_events.csv", index=False)